# 091 — Texto a imagen y condicionamiento

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Con w = 1 se recupera la predicción condicional (ε̃ = 0.26,
dentro del intervalo). Con w > 1 el resultado sale del intervalo [ε_∅, ε_c]:
CFG **extrapola** en la dirección que marca el texto — de ahí la sobresaturación
con guidance alto.

**Ejercicio 2.** El factor 48× (f = 8) explica por qué Stable Diffusion corre en
una GPU de consumo. Con f = 4 el latente es 4× mayor que con f = 8: más fieles
los detalles finos (texto, tramas) pero cada paso cuesta ~4× más.

**Ejercicio 3.** Cambian los valores derivados de la semilla (muestras, métricas
del experimento); no cambian `kind` ni la estructura del contrato. Igual que en
texto-a-imagen: la semilla fija el ruido inicial, el prompt fija la semántica.


In [ ]:
# Ejercicio 1 — CFG a mano
eps_incond = 0.10
eps_cond = 0.26
for w in (1.0, 3.0, 7.5):
    eps_tilde = eps_incond + w * (eps_cond - eps_incond)
    fuera = not (eps_incond <= eps_tilde <= eps_cond)
    print(f"w={w:>4}: eps_tilde = {eps_tilde:.3f}  (fuera del intervalo: {fuera})")


In [ ]:
# Ejercicio 2 — ahorro del espacio latente
pixeles = 512 * 512 * 3
latente_f8 = 64 * 64 * 4
latente_f4 = 128 * 128 * 4
print(f"pixeles      : {pixeles:>9,} valores por paso")
print(f"latente f=8  : {latente_f8:>9,}  -> factor {pixeles / latente_f8:.0f}x")
print(f"latente f=4  : {latente_f4:>9,}  -> factor {pixeles / latente_f4:.0f}x")
print(f"f=4 cuesta {latente_f4 / latente_f8:.0f}x mas que f=8 por paso")


**Ejercicio 4.** El contrato mínimo se valida sin asumir valores internos:


In [ ]:
result = run_lab("generation", seed=91)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)


## Reflexión

1. ¿Por qué CFG exige que el modelo se haya entrenado también con la condición vacía ∅ (dropout del prompt), y qué pasaría si nunca hubiera visto ejemplos incondicionales?
2. Subir w de 7.5 a 20 no da "más obediencia gratis": ¿qué efectos degenerativos aparecen y cómo los explica el hecho de que CFG extrapola fuera del intervalo [ε_∅, ε_c]?
3. Un texto pequeño ilegible dentro de la imagen, ¿es culpa de la U-Net de difusión o del decoder del VAE? Diseña un experimento que lo distinga (pista: reconstruye una imagen real con el autoencoder, sin difusión).
